In [1]:
from google.colab import files
import os

for f in ['repos.csv', 'faible.csv', 'forte.csv', 'model.h']:
    if os.path.exists(f): os.remove(f)

print(" Cliquez sur 'Choisir des fichiers' et sélectionnez les 3 fichiers")
uploaded = files.upload()

 Cliquez sur 'Choisir des fichiers' et sélectionnez les 3 fichiers


Saving faible.csv to faible.csv
Saving forte.csv to forte.csv
Saving repos.csv to repos.csv


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import os

# --- PARAMÈTRES ---
SEQ_LENGTH = 50   # 1 seconde d'enregistrement
CLASSES = ["repos", "faible", "forte"]

# --- FONCTION DE CHARGEMENT INTELLIGENTE ---
def load_and_center_data(filename, label):
    if not os.path.exists(filename):
        print(f" Fichier {filename} manquant !"); return np.array([]), np.array([])

    # 1. On cherche où commencent les données (ligne aX,aY,aZ)
    skip = 0
    try:
        with open(filename, 'r', encoding='utf-8', errors='ignore') as f:
            for i, line in enumerate(f):
                if "aX,aY,aZ" in line: skip = i; break

        # 2. Lecture propre
        df = pd.read_csv(filename, skiprows=skip)
        # On ne garde que les lignes avec des chiffres valides
        df = df[pd.to_numeric(df['aX'], errors='coerce').notnull()]
        data = df[['aX', 'aY', 'aZ']].values.astype(float)
    except Exception as e:
        print(f"Erreur lecture {filename}: {e}"); return np.array([]), np.array([])

    X, y = [], []
    # 3. Découpage + CENTRAGE (Suppression de la gravité)
    # On avance de 50 par 50 (pas de chevauchement pour simplifier)
    for i in range(0, len(data) - SEQ_LENGTH, 50):
        window = data[i:i+SEQ_LENGTH]

        # ASTUCE : On soustrait la moyenne de la fenêtre
        # Ainsi, [0.01, 0.02, 1.00] devient [0, 0, 0] (environ)
        # L'IA ne voit plus que les vibrations !
        centered_window = window - np.mean(window, axis=0)

        X.append(centered_window)
        y.append(label)

    return np.array(X), np.array(y)

print("--- Traitement des données (Mode Anti-Gravité) ---")
X0, y0 = load_and_center_data('repos.csv', 0)
X1, y1 = load_and_center_data('faible.csv', 1)
X2, y2 = load_and_center_data('forte.csv', 2)

if len(X0) == 0:
    print(" ERREUR : Vos fichiers CSV sont vides ou illisibles.")
else:
    # Fusion des données
    X = np.concatenate([X0, X1, X2])
    y = np.concatenate([y0, y1, y2])

    # Mélange aléatoire
    indices = np.arange(X.shape[0])
    np.random.shuffle(indices)
    X, y = X[indices], y[indices]

    print(f" Données prêtes : {len(X)} exemples. Entraînement en cours...")

    # --- CRÉATION DU MODÈLE ---
    model = tf.keras.Sequential([
        layers.Flatten(input_shape=(SEQ_LENGTH, 3)),
        layers.Dense(32, activation='relu'),
        layers.Dense(16, activation='relu'),
        layers.Dense(3, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    # On entraine 40 fois pour être sûr
    model.fit(X, y, epochs=40, batch_size=8, verbose=0)
    print(" Entraînement terminé.")

    # --- EXPORTATION EN C++ ---
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = converter.convert()

    # Fonction pour convertir en hexa
    def hex_to_c_array(data, model_name="model_data"):
        c_str = "#include <cstddef>\n"
        c_str += "const unsigned char " + model_name + "[] = {"
        hex_array = [f"0x{val:02x}" for val in data]
        c_str += ", ".join(hex_array)
        c_str += "};\n"
        c_str += "const int " + model_name + "_len = " + str(len(data)) + ";"
        return c_str

    with open('model.h', 'w') as f:
        f.write(hex_to_c_array(tflite_model))

    print("\n SUCCES ! Téléchargez le fichier 'model.h' dans le menu")

--- Traitement des données (Mode Anti-Gravité) ---
 Données prêtes : 309 exemples. Entraînement en cours...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


 Entraînement terminé.
Saved artifact at '/tmp/tmp10m5r7rs'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 50, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  138817764525456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138817764526416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138817764525840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138817764526224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138817764525264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  138817764524496: TensorSpec(shape=(), dtype=tf.resource, name=None)

 SUCCES ! Téléchargez le fichier 'model.h' dans le menu
